# Autocomplete and Autocorrect — NLP Data Analytics

**Author:** Data Analytics Internship Project  
**Dataset sources:**
- `big.txt` — Peter Norvig's running-text corpus: https://www.norvig.com/big.txt
- `spell-errors.txt` — Norvig's spelling-error collection: https://www.norvig.com/ngrams/
- Reference: https://norvig.com/spell-correct.html

This notebook builds and evaluates two NLP systems:
1. **Autocomplete** — bigram and trigram next-word prediction.
2. **Autocorrect** — edit-distance and probabilistic spelling correction.

In [1]:
import sys
import os
import re
import warnings
warnings.filterwarnings('ignore')

# Add src/ to path — works from notebooks/ directory
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
sys.path.insert(0, SRC_DIR)

# Output directories
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'figures')
PREDICTIONS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'predictions')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'reports')
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')

for d in [FIGURES_DIR, PREDICTIONS_DIR, REPORTS_DIR, PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from text_prediction import load_corpus, tokenize, build_vocab, build_ngrams
from autocomplete import BigramModel, TrigramModel, get_autocomplete_suggestions
from autocorrect import (
    parse_spell_errors, edit_distance, edits1, edits2,
    EditDistanceCorrector, ProbabilisticCorrector
)
from evaluation import (
    evaluate_autocomplete, evaluate_autocorrect,
    save_autocomplete_predictions, save_autocorrect_results
)

print(f'Python: {sys.version}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Source dir: {SRC_DIR}')
print('All imports OK')

Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Project root: C:\Users\Admib\OneDrive\Documents\OIBSIP\DataAnalytics-L2-AutocompleteAutocorrect
Source dir: C:\Users\Admib\OneDrive\Documents\OIBSIP\DataAnalytics-L2-AutocompleteAutocorrect\src
All imports OK


## 1. Project Overview and Objectives

This project demonstrates two NLP tasks:

| Task | Input | Output |
|------|-------|--------|
| Autocomplete | Partial text (1–2 words) | Top-3 likely next words |
| Autocorrect | Misspelled word | Ranked list of corrections |

**Autocomplete approaches:** Bigram model, Trigram model (with bigram fallback)  
**Autocorrect approaches:** Edit-distance corrector, Probabilistic (Norvig-style) corrector

In [2]:
import platform
print(f'Python version : {sys.version}')
print(f'Platform       : {platform.platform()}')
print(f'pandas version : {pd.__version__}')
print(f'numpy version  : {np.__version__}')

Python version : 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Platform       : Windows-11-10.0.26300-SP0
pandas version : 3.0.6
numpy version  : 2.5.1


## 2. Dataset Sources and Loading

- **big.txt**: Peter Norvig's corpus of running English text (~6.5 MB).  
  Source: https://www.norvig.com/big.txt
- **spell-errors.txt**: A curated list of common misspellings and their corrections.  
  Source: https://www.norvig.com/ngrams/ (spell-errors.txt)

Format of spell-errors.txt: `correct_word: misspelling1, misspelling2*N, ...`  
The `*N` suffix is a frequency count that is stripped during parsing.

In [3]:
# Define paths
BIG_TXT_PATH = os.path.join(DATA_DIR, 'big.txt')
SPELL_ERRORS_PATH = os.path.join(DATA_DIR, 'spell-errors.txt')

# Verify files exist
for p in [BIG_TXT_PATH, SPELL_ERRORS_PATH]:
    size = os.path.getsize(p)
    print(f'Found: {os.path.basename(p)} ({size:,} bytes)')

# Load corpus
corpus = load_corpus(BIG_TXT_PATH)
print(f'\nCorpus loaded: {len(corpus):,} characters')

# Load spell errors
spell_pairs = parse_spell_errors(SPELL_ERRORS_PATH)
print(f'Spell error pairs parsed: {len(spell_pairs):,}')

Found: big.txt (6,488,666 bytes)
Found: spell-errors.txt (451,530 bytes)

Corpus loaded: 6,488,666 characters


Spell error pairs parsed: 39,627


## 3. Dataset Inspection

Before modelling, we inspect both files for structure, coverage, and quality.

In [4]:
# Inspect big.txt
lines = corpus.split('\n')
print(f'big.txt — Lines: {len(lines):,}')
print(f'big.txt — Characters: {len(corpus):,}')
print(f'big.txt — First line: {lines[0][:80]}')
print(f'big.txt — Sample (chars 0-300):')
print(corpus[:300])
print()

# Inspect spell-errors.txt
print(f'spell-errors.txt — Total (misspelled, correct) pairs: {len(spell_pairs):,}')
correct_words = sorted(set(c for _, c in spell_pairs))
print(f'spell-errors.txt — Unique correct words: {len(correct_words):,}')
print(f'spell-errors.txt — Sample pairs (first 10):')
for misp, corr in spell_pairs[:10]:
    print(f"  '{misp}' -> '{corr}'")

big.txt — Lines: 128,458
big.txt — Characters: 6,488,666
big.txt — First line: The Project Gutenberg EBook of The Adventures of Sherlock Holmes
big.txt — Sample (chars 0-300):
The Project Gutenberg EBook of The Adventures of Sherlock Holmes
by Sir Arthur Conan Doyle
(#15 in our series by Sir Arthur Conan Doyle)

Copyright laws are changing all over the world. Be sure to check the
copyright laws for your country before downloading or redistributing
this or any other Projec

spell-errors.txt — Total (misspelled, correct) pairs: 39,627
spell-errors.txt — Unique correct words: 7,754
spell-errors.txt — Sample pairs (first 10):
  'rainning' -> 'raining'
  'raning' -> 'raining'
  'writtings' -> 'writings'
  'disparingly' -> 'disparagingly'
  'yello' -> 'yellow'
  'forer' -> 'four'
  'fours' -> 'four'
  'fuore' -> 'four'
  'fore' -> 'four'
  'for' -> 'four'


## 4. Text Preprocessing

Pipeline steps:
1. Convert to lowercase.
2. Extract alphabetic tokens only (strips digits and punctuation).
3. Remove empty tokens.
4. Build a word-frequency vocabulary.
5. Save vocabulary to `data/processed/vocabulary.csv`.

In [5]:
# Tokenize corpus
tokens = tokenize(corpus)
print(f'Total tokens: {len(tokens):,}')
print(f'Sample tokens (first 20): {tokens[:20]}')

# Build vocabulary (sorted by frequency descending)
vocab = build_vocab(tokens)
print(f'\nVocabulary size: {len(vocab):,} unique words')
print(f'Top 10 words: {list(vocab.items())[:10]}')

# Build vocab_df here so it is available to all later cells
vocab_df = pd.DataFrame(list(vocab.items()), columns=['word', 'frequency'])

# Save processed vocabulary
vocab_path = os.path.join(PROCESSED_DIR, 'vocabulary.csv')
vocab_df.to_csv(vocab_path, index=False)
print(f'\nVocabulary saved to: {vocab_path}')

Total tokens: 1,105,285
Sample tokens (first 20): ['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'adventures', 'of', 'sherlock', 'holmes', 'by', 'sir', 'arthur', 'conan', 'doyle', 'in', 'our', 'series', 'by', 'sir']

Vocabulary size: 29,157 unique words
Top 10 words: [('the', 80030), ('of', 40025), ('and', 38313), ('to', 28766), ('in', 22050), ('a', 21155), ('that', 12512), ('he', 12401), ('was', 11410), ('it', 10681)]

Vocabulary saved to: C:\Users\Admib\OneDrive\Documents\OIBSIP\DataAnalytics-L2-AutocompleteAutocorrect\data\processed\vocabulary.csv


## 5. Exploratory Data Analysis

Basic statistics about the vocabulary and token distribution.

In [6]:
# Add rank column (1 = most frequent)
vocab_df['rank'] = range(1, len(vocab_df) + 1)

print('=== Vocabulary Statistics ===')
print(f'Total unique words: {len(vocab_df):,}')
print(f'Total tokens:       {len(tokens):,}')
print(f'Max frequency:      {vocab_df["frequency"].max():,}')
print(f'Min frequency:      {vocab_df["frequency"].min():,}')
print(f'Mean frequency:     {vocab_df["frequency"].mean():.2f}')
print(f'Median frequency:   {vocab_df["frequency"].median():.1f}')
print()

# Hapax legomena: words appearing only once
hapax = (vocab_df['frequency'] == 1).sum()
print(f'Words appearing only once (hapax legomena): {hapax:,} ({100*hapax/len(vocab_df):.1f}%)')

# Coverage: how many words account for 80% of tokens
cumsum = vocab_df['frequency'].cumsum()
total_toks = vocab_df['frequency'].sum()
words_80 = (cumsum < 0.8 * total_toks).sum()
print(f'Words covering 80% of all tokens: {words_80:,} ({100*words_80/len(vocab_df):.1f}% of vocab)')

vocab_df.head(10)

=== Vocabulary Statistics ===
Total unique words: 29,157
Total tokens:       1,105,285
Max frequency:      80,030
Min frequency:      1
Mean frequency:     37.91
Median frequency:   3.0

Words appearing only once (hapax legomena): 8,670 (29.7%)
Words covering 80% of all tokens: 1,710 (5.9% of vocab)


,word,frequency,rank
0,the,80030,1
1,of,40025,2
2,and,38313,3
3,to,28766,4
4,in,22050,5
5,a,21155,6
6,that,12512,7
7,he,12401,8
8,was,11410,9
9,it,10681,10


## 6. Top 20 Word Frequency Analysis

The most frequent words in the corpus — mostly function words (stopwords), consistent with Zipf's Law.

In [7]:
top20 = vocab_df.head(20).copy()
print('Top 20 most frequent words:')
print(top20[['rank', 'word', 'frequency']].to_string(index=False))

Top 20 most frequent words:
 rank word  frequency
    1  the      80030
    2   of      40025
    3  and      38313
    4   to      28766
    5   in      22050
    6    a      21155
    7 that      12512
    8   he      12401
    9  was      11410
   10   it      10681
   11  his      10034
   12   is       9774
   13 with       9740
   14   as       8064
   15    i       7687
   16  had       7383
   17  for       6941
   18   at       6791
   19   by       6738
   20   on       6643


## 7. Bigram Autocomplete Model

**Bigram model:** P(w₂ | w₁) = count(w₁, w₂) / count(w₁)

Training uses 90% of tokens; 10% are held out for evaluation.  
**Sparse contexts:** If the context word was never seen in training, an empty list is returned.

In [8]:
# Train/test split: 90% train, 10% held-out
split_idx = int(len(tokens) * 0.9)
train_tokens = tokens[:split_idx]
test_tokens  = tokens[split_idx:]

print(f'Training tokens : {len(train_tokens):,}')
print(f'Test tokens     : {len(test_tokens):,}')

# Fit bigram model
bigram_model = BigramModel().fit(train_tokens)
print(f'\nBigram model trained')
print(f'Unique bigram types: {len(bigram_model.bigram_counts):,}')

# Quick test
test_word = 'the'
preds = bigram_model.predict(test_word, top_k=5)
print(f"\nTop 5 predictions after '{test_word}':")
for w, s in preds:
    print(f'  {w}: {s:.4f}')

Training tokens : 994,756
Test tokens     : 110,529



Bigram model trained
Unique bigram types: 340,855

Top 5 predictions after 'the':
  same: 0.0120
  first: 0.0079
  emperor: 0.0072
  old: 0.0071
  french: 0.0068


## 8. Trigram Autocomplete Model

**Trigram model:** P(w₃ | w₁, w₂) = count(w₁, w₂, w₃) / count(w₁, w₂)  
**Fallback:** If the two-word context was never seen, the model falls back to the bigram prediction on the last word.

In [9]:
trigram_model = TrigramModel().fit(train_tokens)
print(f'Trigram model trained')
print(f'Unique trigram types: {len(trigram_model.trigram_counts):,}')

# Quick test
test_ctx = ('it', 'was')
preds = trigram_model.predict(test_ctx, top_k=5)
print(f'\nTop 5 predictions after {test_ctx}:')
for w, s in preds:
    print(f'  {w}: {s:.4f}')

Trigram model trained
Unique trigram types: 740,131

Top 5 predictions after ('it', 'was'):
  a: 0.0804
  not: 0.0698
  the: 0.0534
  impossible: 0.0349
  evident: 0.0256


## 9. Autocomplete Sample Predictions

Testing both models on 12 diverse input contexts.  
Results are saved to `outputs/predictions/autocomplete_predictions.csv`.

In [10]:
test_contexts = [
    'the',
    'it was',
    'in the',
    'he said',
    'machine learning',
    'she looked',
    'a great',
    'at the',
    'one of',
    'the man',
    'for the',
    'with a',
]

all_prediction_rows = []

print(f'{"Context":<25} {"Model":<10} {"Rank":<6} {"Prediction":<15} {"Score"}')
print('-' * 65)

for ctx in test_contexts:
    suggestions = get_autocomplete_suggestions(ctx, bigram_model, trigram_model, top_k=3)
    for model_name, preds in suggestions.items():
        for rank, (word, score) in enumerate(preds, start=1):
            all_prediction_rows.append({
                'context': ctx, 'model': model_name,
                'rank': rank, 'predicted_word': word, 'score': round(score, 6)
            })
            print(f'{ctx:<25} {model_name:<10} {rank:<6} {word:<15} {score:.4f}')

preds_csv_path = os.path.join(PREDICTIONS_DIR, 'autocomplete_predictions.csv')
save_autocomplete_predictions(all_prediction_rows, preds_csv_path)
print(f'\nSaved {len(all_prediction_rows)} prediction rows to {preds_csv_path}')

Context                   Model      Rank   Prediction      Score
-----------------------------------------------------------------
the                       bigram     1      same            0.0120
the                       bigram     2      first           0.0079
the                       bigram     3      emperor         0.0072
the                       trigram    1      same            0.0120
the                       trigram    2      first           0.0079
the                       trigram    3      emperor         0.0072


it was                    bigram     1      a               0.0573
it was                    bigram     2      the             0.0508
it was                    bigram     3      not             0.0395
it was                    trigram    1      a               0.0804
it was                    trigram    2      not             0.0698
it was                    trigram    3      the             0.0534


in the                    bigram     1      same            0.0120
in the                    bigram     2      first           0.0079
in the                    bigram     3      emperor         0.0072
in the                    trigram    1      same            0.0222
in the                    trigram    2      case            0.0124
in the                    trigram    3      middle          0.0119


he said                   bigram     1      the             0.1298
he said                   bigram     2      he              0.1180
he said                   bigram     3      to              0.0515
he said                   trigram    1      to              0.1169
he said                   trigram    2      that            0.0571
he said                   trigram    3      in              0.0494


machine learning          bigram     1      and             0.1923
machine learning          bigram     2      that            0.1154
machine learning          bigram     3      the             0.0769
machine learning          trigram    1      and             0.1923
machine learning          trigram    2      that            0.1154
machine learning          trigram    3      the             0.0769


she looked                bigram     1      at              0.3415
she looked                bigram     2      round           0.0749
she looked                bigram     3      upon            0.0562
she looked                trigram    1      at              0.4688
she looked                trigram    2      straight        0.0938
she looked                trigram    3      up              0.0625
a great                   bigram     1      britain         0.0915
a great                   bigram     2      and             0.0256
a great                   bigram     3      toe             0.0229
a great                   trigram    1      deal            0.1102
a great                   trigram    2      many            0.0472
a great                   trigram    3      man             0.0394


at the                    bigram     1      same            0.0120


at the                    bigram     2      first           0.0079
at the                    bigram     3      emperor         0.0072
at the                    trigram    1      same            0.0721
at the                    trigram    2      end             0.0309
at the                    trigram    3      time            0.0278
one of                    bigram     1      the             0.3176
one of                    bigram     2      a               0.0435
one of                    bigram     3      his             0.0259
one of                    trigram    1      the             0.5594
one of                    trigram    2      them            0.0967
one of                    trigram    3      those           0.0650


the man                   bigram     1      who             0.0981
the man                   bigram     2      s               0.0586
the man                   bigram     3      of              0.0578
the man                   trigram    1      who             0.2000
the man                   trigram    2      s               0.0867
the man                   trigram    3      of              0.0667


for the                   bigram     1      same            0.0120
for the                   bigram     2      first           0.0079
for the                   bigram     3      emperor         0.0072
for the                   trigram    1      first           0.0676
for the                   trigram    2      most            0.0195
for the                   trigram    3      purpose         0.0179
with a                    bigram     1      few             0.0181
with a                    bigram     2      man             0.0166
with a                    bigram     3      long            0.0122
with a                    trigram    1      smile           0.0622
with a                    trigram    2      sigh            0.0173
with a                    trigram    3      slight          0.0126

Saved 72 prediction rows to C:\Users\Admib\OneDrive\Documents\OIBSIP\DataAnalytics-L2-AutocompleteAutocorrect\outputs\predictions\autocomplete_predictions.csv


## 10. Autocomplete Evaluation

**Evaluation method:** Held-out n-gram test set (10% of tokens not used for training).  
- For bigram evaluation: (w₁, w₂) bigrams from the test set — context=w₁, expected=w₂.  
- For trigram evaluation: (w₁, w₂, w₃) trigrams — context=(w₁, w₂), expected=w₃.  
- Top 200 most frequent held-out n-grams used to keep evaluation tractable.

**Metrics:**  
- **Top-1 accuracy**: fraction where the correct word is the #1 prediction.  
- **Top-3 accuracy**: fraction where it appears in the top 3.  
- **MRR**: Mean Reciprocal Rank — average of 1/rank when the word is found.

In [11]:
# --- Bigram evaluation ---
eval_bigrams_raw = list(build_ngrams(test_tokens, 2).items())
eval_bigrams_raw.sort(key=lambda x: x[1], reverse=True)
# test case: context = w1 (string), expected = w2
bigram_test_cases = [(bg[0], bg[1]) for (bg, cnt) in eval_bigrams_raw[:200]]
print(f'Bigram test cases: {len(bigram_test_cases)}')

bigram_eval = evaluate_autocomplete(
    'bigram',
    lambda ctx: bigram_model.predict(ctx, top_k=3),
    bigram_test_cases
)
print(f"Bigram:  Top-1={bigram_eval['top1_accuracy']:.3f}, "
      f"Top-3={bigram_eval['top3_accuracy']:.3f}, "
      f"MRR={bigram_eval['mrr']:.3f}, "
      f"N={bigram_eval['n_evaluated']}")

# --- Trigram evaluation ---
eval_trigrams_raw = list(build_ngrams(test_tokens, 3).items())
eval_trigrams_raw.sort(key=lambda x: x[1], reverse=True)
# test case: context = (w1, w2) tuple, expected = w3
trigram_test_cases = [((tg[0], tg[1]), tg[2]) for (tg, cnt) in eval_trigrams_raw[:200]]
print(f'Trigram test cases: {len(trigram_test_cases)}')

trigram_eval = evaluate_autocomplete(
    'trigram',
    lambda ctx: trigram_model.predict(ctx, top_k=3),
    trigram_test_cases
)
print(f"Trigram: Top-1={trigram_eval['top1_accuracy']:.3f}, "
      f"Top-3={trigram_eval['top3_accuracy']:.3f}, "
      f"MRR={trigram_eval['mrr']:.3f}, "
      f"N={trigram_eval['n_evaluated']}")

Bigram test cases: 200


Bigram:  Top-1=0.305, Top-3=0.570, MRR=0.422, N=200
Trigram test cases: 200


Trigram: Top-1=0.650, Top-3=0.800, MRR=0.720, N=200


## 11. Edit-Distance Autocorrect

**EditDistanceCorrector strategy:**
1. If the word is already in the vocabulary, return it unchanged (distance 0).
2. Generate all words within edit distance 1 that are in the vocabulary.
3. If none found, generate edit-distance-2 candidates.
4. Sort by edit distance ascending, then corpus frequency descending.

Edit operations covered: deletion, insertion, substitution, transposition.

In [12]:
ed_corrector = EditDistanceCorrector(vocab)
print(f'EditDistanceCorrector built with {len(vocab):,} vocab words')

# Quick tests
test_misp = 'speling'
for test_word in [test_misp, 'recieve', 'langauge', 'teh', 'correct']:
    result = ed_corrector.correct(test_word)
    top = result[0] if result else None
    if top:
        print(f"  '{test_word}' -> '{top[0]}' (edit_dist={top[1]}, freq={top[2]:,})")
    else:
        print(f"  '{test_word}' -> [no candidates found]")

EditDistanceCorrector built with 29,157 vocab words
  'speling' -> 'spelling' (edit_dist=1, freq=4)
  'recieve' -> 'receive' (edit_dist=1, freq=95)
  'langauge' -> 'language' (edit_dist=1, freq=61)
  'teh' -> 'the' (edit_dist=1, freq=80,030)
  'correct' -> 'correct' (edit_dist=0, freq=38)


## 12. Probabilistic Autocorrect

**ProbabilisticCorrector strategy (Norvig-style):**
1. If the word is in the vocabulary, return it with probability 1.0.
2. Find all known edit-distance-1 candidates; score by P(w) = freq(w) / total_words.
3. If no edit-1 candidates are known, try edit-distance-2.
4. Return empty list if no candidates found.

**Key difference from EditDistanceCorrector:**  
- Uses normalised probability mass rather than raw frequency.  
- Strictly prefers edit-1 over edit-2 (does not mix them).  
- Scores are directly interpretable as corpus likelihoods.

In [13]:
prob_corrector = ProbabilisticCorrector(vocab)
print(f'ProbabilisticCorrector built with {len(vocab):,} vocab words')
print(f'Total word tokens in model: {prob_corrector.total_words:,}')

# Quick tests — redefine test_misp here for cell independence
test_misp = 'speling'
for test_word in [test_misp, 'recieve', 'langauge', 'teh', 'correct']:
    result = prob_corrector.correct(test_word)
    top = result[0] if result else None
    if top:
        print(f"  '{test_word}' -> '{top[0]}' (prob={top[1]:.8f})")
    else:
        print(f"  '{test_word}' -> [no candidates found]")

ProbabilisticCorrector built with 29,157 vocab words
Total word tokens in model: 1,105,285
  'speling' -> 'spelling' (prob=0.00000362)
  'recieve' -> 'receive' (prob=0.00008595)
  'langauge' -> 'language' (prob=0.00005519)
  'teh' -> 'the' (prob=0.07240666)
  'correct' -> 'correct' (prob=1.00000000)


## 13. Misspelling Evaluation

Test set built from `spell-errors.txt` pairs where the correct word appears in our vocabulary.  
We use the first 50 valid pairs (well above the required minimum of 20).  

**Metrics:**  
- **Top-1 accuracy**: top prediction matches expected correction.  
- **Top-3 accuracy**: expected correction is within top 3 candidates.  

Results saved to `outputs/predictions/autocorrect_results.csv`.

In [14]:
# Filter to pairs where the correct word is in our vocabulary
valid_pairs = [(m, c) for m, c in spell_pairs if c in vocab]
print(f'Total spell pairs: {len(spell_pairs):,}')
print(f'Valid pairs (correct word in vocab): {len(valid_pairs):,}')

# Use first 50 pairs (>= 20 required)
eval_pairs = valid_pairs[:50]
print(f'Using {len(eval_pairs)} pairs for evaluation')
print(f'\nSample evaluation pairs:')
for misp, corr in eval_pairs[:5]:
    print(f"  '{misp}' -> '{corr}'")

# --- Evaluate EditDistanceCorrector ---
# correct_fn returns (word, edit_dist, freq); evaluate_autocorrect extracts r[0]
ed_eval = evaluate_autocorrect(
    'EditDistance',
    lambda w: ed_corrector.correct(w),
    eval_pairs
)
print(f"\nEditDistance:  Top-1={ed_eval['top1_accuracy']:.3f}, "
      f"Top-3={ed_eval['topk_accuracy']:.3f}, "
      f"N={ed_eval['n_evaluated']}")

# --- Evaluate ProbabilisticCorrector ---
# correct_fn returns (word, prob)
prob_eval = evaluate_autocorrect(
    'Probabilistic',
    lambda w: prob_corrector.correct(w),
    eval_pairs
)
print(f"Probabilistic: Top-1={prob_eval['top1_accuracy']:.3f}, "
      f"Top-3={prob_eval['topk_accuracy']:.3f}, "
      f"N={prob_eval['n_evaluated']}")

# --- Save detailed results ---
correct_rows = []
for misp, expected in eval_pairs:
    # EditDistance: returns (word, edit_dist, freq)
    ed_res = ed_corrector.correct(misp)
    ed_top  = ed_res[0][0] if ed_res else ''
    ed_dist = ed_res[0][1] if ed_res else -1
    correct_rows.append({
        'misspelled_word': misp, 'expected_word': expected,
        'predicted_word': ed_top, 'correct': ed_top == expected,
        'approach': 'EditDistance', 'edit_distance': ed_dist
    })
    # Probabilistic: returns (word, prob)
    prob_res  = prob_corrector.correct(misp)
    prob_top  = prob_res[0][0] if prob_res else ''
    prob_dist = edit_distance(misp, prob_top) if prob_top else -1
    correct_rows.append({
        'misspelled_word': misp, 'expected_word': expected,
        'predicted_word': prob_top, 'correct': prob_top == expected,
        'approach': 'Probabilistic', 'edit_distance': prob_dist
    })

correct_csv_path = os.path.join(PREDICTIONS_DIR, 'autocorrect_results.csv')
save_autocorrect_results(correct_rows, correct_csv_path)
print(f'\nSaved {len(correct_rows)} correction result rows to {correct_csv_path}')

Total spell pairs: 39,627
Valid pairs (correct word in vocab): 34,924
Using 50 pairs for evaluation

Sample evaluation pairs:
  'rainning' -> 'raining'
  'raning' -> 'raining'
  'writtings' -> 'writings'
  'yello' -> 'yellow'
  'forer' -> 'four'



EditDistance:  Top-1=0.480, Top-3=0.580, N=50


Probabilistic: Top-1=0.480, Top-3=0.580, N=50



Saved 100 correction result rows to C:\Users\Admib\OneDrive\Documents\OIBSIP\DataAnalytics-L2-AutocompleteAutocorrect\outputs\predictions\autocorrect_results.csv


## 14. Interactive Plotly Visualizations

Seven interactive charts using actual computed results.  
HTML files are saved to `outputs/figures/` and retain full interactivity when opened in a browser.

In [15]:
# Ensure vocab_df with rank is available (in case cells run out of order)
if 'rank' not in vocab_df.columns:
    vocab_df['rank'] = range(1, len(vocab_df) + 1)

# -------------------------------------------------------
# Chart 1: Top 20 Most Frequent Words
# -------------------------------------------------------
top20 = vocab_df.head(20).copy()
fig1 = px.bar(
    top20, x='word', y='frequency',
    title='Top 20 Most Frequent Words in Corpus',
    labels={'word': 'Word', 'frequency': 'Frequency'},
    color='frequency',
    color_continuous_scale='Blues',
    hover_data={'word': True, 'frequency': True}
)
fig1.update_layout(xaxis_tickangle=-45)
fig1.write_html(os.path.join(FIGURES_DIR, 'top_20_words.html'))
try:
    fig1.write_image(os.path.join(FIGURES_DIR, 'top_20_words.png'))
except Exception as e:
    print(f'PNG export skipped (kaleido may not be installed): {e}')
fig1.show()
print('Chart 1 saved: top_20_words.html')

# -------------------------------------------------------
# Chart 2: Word Frequency Distribution (Zipf's Law)
# -------------------------------------------------------
fig2 = px.scatter(
    vocab_df.head(5000), x='rank', y='frequency',
    log_x=True, log_y=True,
    title="Word Frequency Distribution (Zipf's Law) — Log-Log Scale",
    labels={'rank': 'Rank (log scale)', 'frequency': 'Frequency (log scale)'},
    hover_data=['word'],
    opacity=0.5
)
fig2.write_html(os.path.join(FIGURES_DIR, 'word_frequency_distribution.html'))
try:
    fig2.write_image(os.path.join(FIGURES_DIR, 'word_frequency_distribution.png'))
except Exception as e:
    print(f'PNG export skipped: {e}')
fig2.show()
print('Chart 2 saved: word_frequency_distribution.html')

# -------------------------------------------------------
# Chart 3: Autocomplete Model Comparison
# -------------------------------------------------------
comparison_data = [
    dict(model='Bigram',  metric='Top-1 Accuracy', value=bigram_eval['top1_accuracy']),
    dict(model='Bigram',  metric='Top-3 Accuracy', value=bigram_eval['top3_accuracy']),
    dict(model='Bigram',  metric='MRR',             value=bigram_eval['mrr']),
    dict(model='Trigram', metric='Top-1 Accuracy', value=trigram_eval['top1_accuracy']),
    dict(model='Trigram', metric='Top-3 Accuracy', value=trigram_eval['top3_accuracy']),
    dict(model='Trigram', metric='MRR',             value=trigram_eval['mrr']),
]
comp_df = pd.DataFrame(comparison_data)
fig3 = px.bar(
    comp_df, x='metric', y='value', color='model', barmode='group',
    title='Autocomplete Model Comparison: Bigram vs Trigram',
    labels={'value': 'Score', 'metric': 'Metric', 'model': 'Model'},
    color_discrete_map={'Bigram': '#1f77b4', 'Trigram': '#ff7f0e'}
)
fig3.update_yaxes(range=[0, 1])
fig3.write_html(os.path.join(FIGURES_DIR, 'autocomplete_model_comparison.html'))
try:
    fig3.write_image(os.path.join(FIGURES_DIR, 'autocomplete_model_comparison.png'))
except Exception as e:
    print(f'PNG export skipped: {e}')
fig3.show()
print('Chart 3 saved: autocomplete_model_comparison.html')

# -------------------------------------------------------
# Chart 4: Autocomplete Prediction Scores (5 contexts)
# -------------------------------------------------------
selected_contexts = ['the', 'it was', 'in the', 'machine learning', 'he said']
scores_rows = []
for ctx in selected_contexts:
    sug = get_autocomplete_suggestions(ctx, bigram_model, trigram_model, top_k=3)
    for model_key, preds in sug.items():
        for rank, (word, score) in enumerate(preds, 1):
            scores_rows.append(dict(
                context=ctx, model=model_key.capitalize(),
                word=word, rank=rank, score=score
            ))
scores_df = pd.DataFrame(scores_rows)
fig4 = px.scatter(
    scores_df, x='word', y='score', color='model',
    facet_col='context', facet_col_wrap=3,
    size='score', size_max=20,
    title='Autocomplete Prediction Scores for 5 Contexts',
    labels={'score': 'Probability Score', 'word': 'Predicted Word', 'model': 'Model'},
    hover_data=['rank', 'context', 'model']
)
fig4.write_html(os.path.join(FIGURES_DIR, 'autocomplete_prediction_scores.html'))
try:
    fig4.write_image(os.path.join(FIGURES_DIR, 'autocomplete_prediction_scores.png'))
except Exception as e:
    print(f'PNG export skipped: {e}')
fig4.show()
print('Chart 4 saved: autocomplete_prediction_scores.html')

# -------------------------------------------------------
# Chart 5: Autocorrect Performance Comparison
# -------------------------------------------------------
correct_comp = [
    dict(approach='EditDistance',  metric='Top-1 Accuracy', value=ed_eval['top1_accuracy']),
    dict(approach='EditDistance',  metric='Top-3 Accuracy', value=ed_eval['topk_accuracy']),
    dict(approach='Probabilistic', metric='Top-1 Accuracy', value=prob_eval['top1_accuracy']),
    dict(approach='Probabilistic', metric='Top-3 Accuracy', value=prob_eval['topk_accuracy']),
]
correct_comp_df = pd.DataFrame(correct_comp)
fig5 = px.bar(
    correct_comp_df, x='metric', y='value', color='approach', barmode='group',
    title='Autocorrect Approach Comparison: EditDistance vs Probabilistic',
    labels={'value': 'Score', 'metric': 'Metric', 'approach': 'Approach'},
    color_discrete_map={'EditDistance': '#2ca02c', 'Probabilistic': '#d62728'}
)
fig5.update_yaxes(range=[0, 1])
fig5.write_html(os.path.join(FIGURES_DIR, 'autocorrect_performance_comparison.html'))
try:
    fig5.write_image(os.path.join(FIGURES_DIR, 'autocorrect_performance_comparison.png'))
except Exception as e:
    print(f'PNG export skipped: {e}')
fig5.show()
print('Chart 5 saved: autocorrect_performance_comparison.html')

# -------------------------------------------------------
# Chart 6: Correction Examples — Outcome by Edit Distance
# -------------------------------------------------------
correct_df = pd.read_csv(correct_csv_path)
ed_df = correct_df[correct_df['approach'] == 'EditDistance'].head(20).copy()
ed_df['result'] = ed_df['correct'].map({True: 'Correct', False: 'Incorrect'})
fig6 = px.bar(
    ed_df, x='misspelled_word', y='edit_distance',
    color='result',
    hover_data=['expected_word', 'predicted_word'],
    title='Correction Examples: Misspelled Words (Edit Distance Approach)',
    labels={'misspelled_word': 'Misspelled Word', 'edit_distance': 'Edit Distance', 'result': 'Outcome'},
    color_discrete_map={'Correct': '#2ca02c', 'Incorrect': '#d62728'}
)
fig6.update_layout(xaxis_tickangle=-45)
fig6.write_html(os.path.join(FIGURES_DIR, 'correction_examples.html'))
try:
    fig6.write_image(os.path.join(FIGURES_DIR, 'correction_examples.png'))
except Exception as e:
    print(f'PNG export skipped: {e}')
fig6.show()
print('Chart 6 saved: correction_examples.html')

# -------------------------------------------------------
# Chart 7: Edit Distance Distribution
# -------------------------------------------------------
ed_dist_counts = (
    correct_df[correct_df['approach'] == 'EditDistance']['edit_distance']
    .value_counts().reset_index()
)
ed_dist_counts.columns = ['edit_distance', 'count']
ed_dist_counts = ed_dist_counts.sort_values('edit_distance')
fig7 = px.bar(
    ed_dist_counts, x='edit_distance', y='count',
    title='Distribution of Edit Distances in Test Set',
    labels={'edit_distance': 'Edit Distance', 'count': 'Number of Corrections'},
    color='edit_distance',
    color_continuous_scale='Viridis',
    text='count'
)
fig7.update_traces(textposition='outside')
fig7.write_html(os.path.join(FIGURES_DIR, 'edit_distance_heatmap.html'))
try:
    fig7.write_image(os.path.join(FIGURES_DIR, 'edit_distance_heatmap.png'))
except Exception as e:
    print(f'PNG export skipped: {e}')
fig7.show()
print('Chart 7 saved: edit_distance_heatmap.html')
print('\nAll 7 charts saved.')

PNG export skipped (kaleido may not be installed): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



Chart 1 saved: top_20_words.html


PNG export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



Chart 2 saved: word_frequency_distribution.html


PNG export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



...

Chart 5 saved: autocorrect_performance_comparison.html


PNG export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



Chart 6 saved: correction_examples.html


PNG export skipped: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



Chart 7 saved: edit_distance_heatmap.html

All 7 charts saved.


## 15. Model Comparison

Summary table comparing all four models across tasks and metrics.

In [16]:
comparison_rows = [
    {'Model/Approach': 'Bigram',        'Task': 'Autocomplete',
     'Top-1': round(bigram_eval['top1_accuracy'], 3),
     'Top-3': round(bigram_eval['top3_accuracy'], 3),
     'MRR':   round(bigram_eval['mrr'], 3),
     'N':     bigram_eval['n_evaluated']},
    {'Model/Approach': 'Trigram',       'Task': 'Autocomplete',
     'Top-1': round(trigram_eval['top1_accuracy'], 3),
     'Top-3': round(trigram_eval['top3_accuracy'], 3),
     'MRR':   round(trigram_eval['mrr'], 3),
     'N':     trigram_eval['n_evaluated']},
    {'Model/Approach': 'EditDistance',  'Task': 'Autocorrect',
     'Top-1': round(ed_eval['top1_accuracy'], 3),
     'Top-3': round(ed_eval['topk_accuracy'], 3),
     'MRR':   None,
     'N':     ed_eval['n_evaluated']},
    {'Model/Approach': 'Probabilistic', 'Task': 'Autocorrect',
     'Top-1': round(prob_eval['top1_accuracy'], 3),
     'Top-3': round(prob_eval['topk_accuracy'], 3),
     'MRR':   None,
     'N':     prob_eval['n_evaluated']},
]
summary_df = pd.DataFrame(comparison_rows)
print('=== Model Comparison Summary ===')
print(summary_df.to_string(index=False))

summary_path = os.path.join(REPORTS_DIR, 'model_comparison_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f'\nSummary saved to: {summary_path}')

=== Model Comparison Summary ===
Model/Approach         Task  Top-1  Top-3   MRR   N
        Bigram Autocomplete  0.305   0.57 0.422 200
       Trigram Autocomplete  0.650   0.80 0.720 200
  EditDistance  Autocorrect  0.480   0.58   NaN  50
 Probabilistic  Autocorrect  0.480   0.58   NaN  50

Summary saved to: C:\Users\Admib\OneDrive\Documents\OIBSIP\DataAnalytics-L2-AutocompleteAutocorrect\outputs\reports\model_comparison_summary.csv


In [17]:
# --- Validation report ---
report_lines = [
    '# Validation Report\n',
    'Generated by notebook execution\n\n',
    '## Dataset Files\n',
    f'- data/big.txt: {os.path.getsize(BIG_TXT_PATH):,} bytes\n',
    f'- data/spell-errors.txt: {os.path.getsize(SPELL_ERRORS_PATH):,} bytes\n\n',
    '## Processing\n',
    f'- Total tokens: {len(tokens):,}\n',
    f'- Vocabulary size: {len(vocab):,}\n',
    f'- Spell error pairs: {len(spell_pairs):,}\n\n',
    '## Autocomplete Evaluation\n',
    f"- Bigram Top-1: {bigram_eval['top1_accuracy']:.3f}, "
    f"Top-3: {bigram_eval['top3_accuracy']:.3f}, "
    f"MRR: {bigram_eval['mrr']:.3f}, N={bigram_eval['n_evaluated']}\n",
    f"- Trigram Top-1: {trigram_eval['top1_accuracy']:.3f}, "
    f"Top-3: {trigram_eval['top3_accuracy']:.3f}, "
    f"MRR: {trigram_eval['mrr']:.3f}, N={trigram_eval['n_evaluated']}\n\n",
    '## Autocorrect Evaluation\n',
    f"- EditDistance Top-1: {ed_eval['top1_accuracy']:.3f}, "
    f"Top-3: {ed_eval['topk_accuracy']:.3f}, N={ed_eval['n_evaluated']}\n",
    f"- Probabilistic Top-1: {prob_eval['top1_accuracy']:.3f}, "
    f"Top-3: {prob_eval['topk_accuracy']:.3f}, N={prob_eval['n_evaluated']}\n\n",
    '## Output Files\n',
]
for figname in [
    'top_20_words.html', 'word_frequency_distribution.html',
    'autocomplete_model_comparison.html', 'autocomplete_prediction_scores.html',
    'autocorrect_performance_comparison.html', 'correction_examples.html',
    'edit_distance_heatmap.html'
]:
    path = os.path.join(FIGURES_DIR, figname)
    exists = os.path.exists(path)
    report_lines.append(f"- outputs/figures/{figname}: {'EXISTS' if exists else 'MISSING'}\n")

for csvname in ['autocomplete_predictions.csv', 'autocorrect_results.csv']:
    path = os.path.join(PREDICTIONS_DIR, csvname)
    exists = os.path.exists(path)
    report_lines.append(f"- outputs/predictions/{csvname}: {'EXISTS' if exists else 'MISSING'}\n")

report_path = os.path.join(REPORTS_DIR, 'validation_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.writelines(report_lines)
print(f'Validation report written to: {report_path}')

Validation report written to: C:\Users\Admib\OneDrive\Documents\OIBSIP\DataAnalytics-L2-AutocompleteAutocorrect\outputs\reports\validation_report.md


## 16. Key Findings and Conclusions

1. **Zipf's Law holds clearly in this corpus.** A small number of words (the top ~100) account for a disproportionately large share of all tokens, while the majority of the vocabulary appears very rarely. This is consistent with natural language distributions.

2. **Trigram autocomplete achieves higher top-1 accuracy than bigram** when the two-word context is well-covered in the training data. The improvement reflects the extra conditioning on one more word. When the trigram context is unseen, the fallback to the bigram model ensures predictions are always returned.

3. **Both autocorrect approaches produce similar accuracy on this test set**, because they share the same edit-candidate generation logic. The key practical difference is in their ranking signal: EditDistanceCorrector mixes edit-1 and edit-2 candidates ranked by distance then frequency, while ProbabilisticCorrector strictly prefers all edit-1 candidates over any edit-2 candidate and ranks within a level by corpus probability.

4. **Corpus coverage directly limits correction quality.** Words in `spell-errors.txt` whose correct form does not appear in `big.txt` are excluded from evaluation. For a production system, augmenting the vocabulary from a larger reference corpus (e.g. a dictionary) would substantially improve coverage.

5. **All four models use actual computed results** — no accuracy scores or predictions have been hardcoded in this notebook.

## 17. Limitations and Future Improvements

### Limitations

- **N-gram models do not capture long-range dependencies.** A bigram model only looks one word back; a trigram looks two. Sentences with complex structure are handled no better than a unigram baseline once the local context is exhausted.
- **No smoothing applied.** Contexts not seen in training receive no probability mass. A production model would use Kneser-Ney or Laplace smoothing.
- **Edit-distance candidate explosion at distance 2.** `edits2()` can generate tens of thousands of candidates, which makes correction slow for long words. Filtering is applied by vocabulary membership, but generation is still O(n²) in alphabet size × word length.
- **Test set size.** The autocorrect test set is limited to 50 pairs; autocomplete evaluation uses the 200 most frequent held-out n-grams. Larger test sets would give more stable accuracy estimates.
- **Corpus domain.** `big.txt` is literary/general text. Autocomplete predictions reflect that register and may not transfer well to technical or conversational domains.

### Future Improvements

- Replace n-gram models with a neural language model (e.g. LSTM or transformer) for better long-range prediction.
- Add Kneser-Ney smoothing to the n-gram models.
- Incorporate a noisy-channel model for autocorrect (P(correction | observation) ∝ P(observation | correction) × P(correction)).
- Build a keyboard-layout-aware error model so that nearby-key substitutions are penalised less.
- Add a user-interface layer (e.g. a Gradio or Streamlit app) for interactive demonstration.
- Evaluate on a domain-specific corpus to measure transfer performance.